In [9]:
import pandas as pd
import numpy as np
import zipfile

import torch
import torchvision
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [10]:
with zipfile.ZipFile('saving_xmas.zip') as zip_ref:
    zip_ref.extractall()

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df.head()

,image_path,number
0,images/image_0214.png,0
1,images/image_0187.png,4
2,images/image_0000.png,0
3,images/image_0262.png,5
4,images/image_0082.png,3


In [11]:
train_df['number'].value_counts()

,count
number,
4,38
5,38
0,36
2,36
3,31
1,21


In [12]:
from sklearn.model_selection import train_test_split

class ImageDataset(Dataset):
    def __init__(self, df, transformer, has_labels):
        self.df = df
        self.transformer = transformer
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image_path = row['image_path']

        image = Image.open(image_path).convert('RGB')
        image = self.transformer(image)

        if self.has_labels:
            return image, np.float32(row['number'])

        return image

#Define transformers
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transformer = transforms.Compose([
    transforms.Resize((300,300)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transformer = transforms.Compose([
    transforms.Resize((300,300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['number'], random_state=42)

train_dataset = ImageDataset(train_df, transformer = train_transformer, has_labels=True)
val_dataset = ImageDataset(val_df, transformer = test_transformer, has_labels=True)
test_dataset = ImageDataset(test_df, transformer = test_transformer, has_labels=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [13]:
#Create a model
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.base = torchvision.models.resnet34(weights='DEFAULT')
        self.base.fc = nn.Identity()

        self.head = nn.Sequential(
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.base(x)
        return self.head(x)

model = Model().to(device)

In [14]:
from torchmetrics import MeanAbsoluteError

mae_metric = MeanAbsoluteError().to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

best_val_mae = 1000.0 #Can't be higher than this, right?

for epoch in range(1, 31):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels.reshape(-1, 1))
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /=len(train_loader)
    #Validation
    model.eval()
    mae_metric.reset()

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            labels = labels.reshape(-1, 1)

            outputs = model(images)
            mae_metric.update(outputs, labels)

    val_mae = mae_metric.compute().item()

    print(f'Epoch {epoch:2d} | Train loss: {train_loss:.4f} | MAE: {val_mae:.4f}')

    #Saving best version
    if val_mae <= best_val_mae:
        best_val_mae = val_mae
        torch.save({
            'model_state_dict': model.state_dict()
            },
            'best_model.pth')
        print('-> Saved weights!')

print(f'Best MAE: {best_val_mae:.4f}')

Epoch  1 | Train loss: 2.2017 | MAE: 1.6726
-> Saved weights!
Epoch  2 | Train loss: 1.3074 | MAE: 1.8836
Epoch  3 | Train loss: 0.7724 | MAE: 3.8201
Epoch  4 | Train loss: 0.6088 | MAE: 2.3837
Epoch  5 | Train loss: 0.5500 | MAE: 1.2817
-> Saved weights!
Epoch  6 | Train loss: 0.5841 | MAE: 0.5689
-> Saved weights!
Epoch  7 | Train loss: 0.4383 | MAE: 0.7505
Epoch  8 | Train loss: 0.4333 | MAE: 0.4846
-> Saved weights!
Epoch  9 | Train loss: 0.3941 | MAE: 0.5224
Epoch 10 | Train loss: 0.4193 | MAE: 0.5084
Epoch 11 | Train loss: 0.5301 | MAE: 0.7631
Epoch 12 | Train loss: 0.4034 | MAE: 0.4993
Epoch 13 | Train loss: 0.3444 | MAE: 0.4618
-> Saved weights!
Epoch 14 | Train loss: 0.3916 | MAE: 0.3510
-> Saved weights!
Epoch 15 | Train loss: 0.3252 | MAE: 0.3659
Epoch 16 | Train loss: 0.3390 | MAE: 0.3208
-> Saved weights!
Epoch 17 | Train loss: 0.3348 | MAE: 0.3969
Epoch 18 | Train loss: 0.3227 | MAE: 0.3249
Epoch 19 | Train loss: 0.2918 | MAE: 0.3995
Epoch 20 | Train loss: 0.2913 | MAE: 0

In [15]:
#Predicting
checkpoint = torch.load("best_model.pth", map_location="cpu")
model = Model().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()
preds = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)

        for output in outputs:
            pred_float = output.item()
            pred_int = int(torch.round(torch.tensor(pred_float)))
            preds.append(pred_int)

In [16]:
output_df = pd.DataFrame({
    'image_path':test_df['image_path'],
    'number':preds
})

output_df.to_csv("submission.csv", index=False)

A good idea was to give bigger images to resnet (discovered accidentally while playing with both efficientnet and resnet)
